In [0]:
import pandas as pd

In [0]:
test = pd.read_parquet("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/nba_model_run/pred_files/test_4c_d8c99075.parquet")

In [0]:
test.head()

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================
PARQUET_PATH = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/nba_model_run/pred_files/test_4c_5c2bd04e.parquet"
N_DECILES = 50   # change this to 10, 20, 50, etc.

CSV_NAME = f"trial19_lift_summary_{N_DECILES}_deciles.csv"

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_parquet(PARQUET_PATH)

required_cols = {"CUSTOMER_ID", "Actual", "Predicted"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in parquet: {missing}")

df = df[["CUSTOMER_ID", "Actual", "Predicted"]].copy()
df["Actual"] = pd.to_numeric(df["Actual"], errors="coerce")
df["Predicted"] = pd.to_numeric(df["Predicted"], errors="coerce")
df = df.dropna(subset=["Actual", "Predicted"]).reset_index(drop=True)

# ============================================================
# LIFT CHART FUNCTION
# ============================================================
def create_lift_chart(actual, predicted, title, n_deciles=10):
    data = pd.DataFrame({
        "Actual": np.asarray(actual).flatten(),
        "Predicted": np.asarray(predicted).flatten()
    })

    data = data.sort_values("Predicted", ascending=False).reset_index(drop=True)

    # Safe number of bins
    q = min(n_deciles, len(data))
    if q < 2:
        raise ValueError("Not enough rows to create deciles.")

    data["_rank"] = np.arange(len(data))
    data["Decile"] = pd.qcut(
        data["_rank"],
        q=q,
        labels=False,
        duplicates="drop"
    ) + 1

    # Scale labels to 10, 20, ..., 100 (or equivalent for q bins)
    data["Decile"] = np.round(data["Decile"] * (100 / q)).astype(int)

    overall_actual_mean = data["Actual"].mean()
    overall_pred_mean = data["Predicted"].mean()

    summary = data.groupby("Decile", as_index=False).agg(
        n=("Actual", "size"),
        actual_mean=("Actual", "mean"),
        pred_mean=("Predicted", "mean")
    )

    summary["actual_lift"] = (
        summary["actual_mean"] / overall_actual_mean if overall_actual_mean > 0 else 0
    )
    summary["predicted_lift"] = (
        summary["pred_mean"] / overall_pred_mean if overall_pred_mean > 0 else 0
    )

    fig, ax = plt.subplots(figsize=(12, 7))

    ax.plot(
        summary["Decile"],
        summary["predicted_lift"],
        marker="o",
        linewidth=2,
        color="steelblue",
        label="Predicted lift"
    )

    ax.plot(
        summary["Decile"],
        summary["actual_lift"],
        marker="s",
        linewidth=2,
        color="orange",
        linestyle="--",
        label="Actual lift"
    )

    ax.set_xlabel("Decile (ranked by Predicted NBRx)")
    ax.set_ylabel("Per-decile lift (mean / overall mean)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

    x_ticks = sorted(summary["Decile"].unique().tolist())
    ax.set_xticks(x_ticks)

    plt.tight_layout()
    plt.show()

    return summary

# ============================================================
# RUN
# ============================================================
summary_df = create_lift_chart(
    actual=df["Actual"],
    predicted=df["Predicted"],
    title=f"Lift Chart - {N_DECILES} Deciles",
    n_deciles=N_DECILES
)

# Show the summary table in notebook
try:
    display(summary_df)
except Exception:
    print(summary_df.to_string(index=False))

# Save CSV in current working directory
summary_df.to_csv(CSV_NAME, index=False)
print(f"CSV saved as: {CSV_NAME}")